## Imports

In [1]:
from langchain_ollama import OllamaEmbeddings
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_ollama import ChatOllama
import os
from dotenv import load_dotenv
import requests
from global_variables import LOCAL_MODEL_NAME, LOCAL_BASE_URL


## Globale Variablen

In [2]:
system_prompt = "Du bist ein Tutor für Vorlesungsinhalte. Beantworte Fragen nur auf Basis des bereitgestellten Kontexts. Erkläre klar, korrekt und verständlich. Wenn Informationen fehlen oder unsicher sind, sage das ausdrücklich. Erfinde nichts und spekuliere nicht. Nutze Fachbegriffe korrekt und erkläre sie kurz, wenn nötig. Gib falls Informationen aus der Fuktion search_lecture_docs entnommen werden das heißt dass die Informationen aus einer Datei kommen, immer den Dateipfad mit an."

LOCAL = True

## Tools

Retreival Tool

In [3]:
# Das Tool stellt Anfragen an die VectorDB und bekommt die entsprechenden Chunks zurück
@tool('search_lecture_docs', description='Retrieves information from Lecture related Documents')
def search_lecture_docs(query: str):
    response = requests.post(
    "http://127.0.0.1:8000/query",
    json={
        "query": query,
        "n": 3
    })
    return response.json()


@tool('get_file_info', description=""" Retrieves all information/chunks from a specific lecture document. IMPORTANT: The `filename` parameter MUST contain ONLY the filename explicitly mentioned by the user. Never pass the user's complete question or any additional text. If the user explicitly refers to a file, extract only the filename including its file extension and pass it as the `filename` parameter. Examples: User: "Was steht in der Datei informationen.txt?" filename: "informationen.txt" User: "Was steht in example.md?" filename: "example.md" User: "Erkläre mir den Inhalt von lecture_03.pdf." filename: "lecture_03.pdf" User: "Was wird in 01-introduction.md über RAG erklärt?" filename: "01-introduction.md" User: "Gib mir die Informationen aus /data/lectures/example.md." filename: "example.md" Do NOT include words such as: - "Was steht in" - "Datei" - "Erkläre" - "Inhalt von" - the user's complete question - the file path, if the user provides a path The filename must be passed exactly as extracted from the user's request, including its extension. This tool should ONLY be used when the user explicitly identifies a specific file. If no specific filename is mentioned, do not use this tool. """)
def get_file_info(filename: str):
    print(f"get_file_info: {filename}")
    response = requests.post(
    "http://127.0.0.1:8000/document",
    json={
        "filename": filename
    })
    return response.json()

## Initialisierungen

In [4]:
# Model für Lokale Ollama Modelle
model_local = ChatOllama(
    base_url=LOCAL_BASE_URL,
    model=LOCAL_MODEL_NAME,
)
# Model für Nvidia NIM API
load_dotenv()
model_api = ChatOpenAI(
    model="meta/llama-3.1-8b-instruct",
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.environ["NVIDIA_API_KEY"],
)

agent = create_agent(model_local if LOCAL else model_api, tools=[search_lecture_docs, get_file_info], system_prompt=system_prompt)

In [5]:
prompt = str(input())
print(prompt + "\n")
result = agent.invoke({"messages": [("user", prompt)]})
print(result["messages"][-1].content)

Was steht in der Datei mit dem Namen example.md

get_file_info: example.md
In der Datei **`example.md`** (Pfad: `data/processed/example.md`) findest du das sogenannte Firmenhandbuch der Beispiel AG. Es enthält die wichtigsten Regelungen und Richtlinien für die tägliche Arbeit und den Betriebsalltag.  
Hier ein Überblick der wichtigsten Abschnitte:

| Abschnitt | Inhalt |
|-----------|--------|
| **Öffnungszeiten** | Büro: Montag‑Freitag, 09:00–17:00 Uhr |
| **IT‑Support** | Passwortprobleme an support@example.com, dringende Störungen telefonisch unter +49 30 12345678; Software‑Installationen nur vom IT‑Team freizugeben |
| **Urlaubsregelung** | 30 Urlaubstage/Jahr, Antrag mindestens 2 Wochen vor Beginn, Resturlaub bis 31 März des Folgejahres |
| **Reisekosten** | Bahn‑2. Klasse erstattungsfähig, Hotelkosten bis 120 €/Nacht, Taxi nur bei dringendem Anlass mit Beleg |
| **Projekt Alpha** | Start 15. Juli 2026, Ansprechpartnerin Maria Schneider, wöchentliche Status‑Meetings montags um 10: